In [ ]:
"""
Statistic Test of HRV and PRV Features 

Compares features from ECG (HRV) and PPG (PRV) across subjects and two methods 
('All Windows', 'Method 2') using Wilcoxon tests. 
Prints how many features are not significantly different (p ≥ 0.05).
"""


import os
import pandas as pd
import re
from utils_statistical_tests import (
    check_normality,
    test_wilcoxon,
    plot_pvalue_heatmap
)

%matplotlib qt


# =============================================================================
# SETUP PATHS
# =============================================================================

# Define the base directory to data
base_path = os.path.join("..", "Data")

methods = {
    "All Windows": os.path.join(base_path, "All Windows"),
    "Method 2": os.path.join(base_path, "Window Removal")
}

# List subjects based on HRV folder of 'All Windows' and sort numerically
subjects_dir = os.path.join(methods["All Windows"], "HRV")
subjects = [f for f in os.listdir(subjects_dir) if f.endswith(".csv")]
subjects = sorted(subjects, key=lambda x: int(re.findall(r'\d+', x)[0]))
subjects = [s.replace(".csv", "") for s in subjects]

# Initialize containers
normality_records = {m: [] for m in methods}
pvalues, effects, distribution = {}, {}, {}


# =============================================================================
# NORMALITY CHECK (Shapiro test)
# =============================================================================

for method_name, method_path in methods.items():
    for subj_name in subjects:
        hrv_path = os.path.join(method_path, "HRV", f"{subj_name}.csv")
        prv_path = os.path.join(method_path, "PRV", f"{subj_name}.csv")

        df_hrv = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
        df_prv = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

        normality_records[method_name].extend(check_normality(df_hrv, df_prv))

    recs = normality_records[method_name]
    ratio = sum([is_norm for _, is_norm in recs]) / len(recs) if recs else 0
    print(f"\nNormality ratio for {method_name}: {ratio:.1%} → "
          f"{'Per-feature Tests' if ratio > 0.7 else 'Wilcoxon Test'}")


# =============================================================================
# STATISTICAL TESTS (Wilcoxon test)
# =============================================================================

for method_name, method_path in methods.items():
    pvalues[method_name], effects[method_name] = {}, {}
    for subj_name in subjects:
        hrv_path = os.path.join(method_path, "HRV", f"{subj_name}.csv")
        prv_path = os.path.join(method_path, "PRV", f"{subj_name}.csv")
        if not os.path.exists(hrv_path) or not os.path.exists(prv_path):
            continue

        df_hrv = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
        df_prv = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

        res = test_wilcoxon(df_hrv, df_prv)
        res = res.set_index('Feature')
        pvalues[method_name][subj_name] = res['p_value']
        effects[method_name][subj_name] = res['Effect size']


# =============================================================================
# SUMMARY & VISUALIZATION
# =============================================================================

for method_name in methods:
    print(f"\n===== {method_name.upper()} =====")
    df_p = pd.DataFrame(pvalues[method_name])
    df_e = pd.DataFrame(effects[method_name])
    plot_pvalue_heatmap(df_p, method_name=method_name)
    nonsig_counts = (df_p >= 0.05).sum(axis=1)
    total = df_p.shape[1]
    print("Non-significant features (p ≥ 0.05):")
    for feat, count in nonsig_counts.items():
        if count > 0:
            print(f"{feat}: {count}/{total}")


    # =============================================================================
    # OPTIONAL: SAVE P-VALUE AND EFFECT SIZE TABLE
    # =============================================================================
    # output_path = "C:/Users/ilari/Downloads"
    # df_p.to_csv(os.path.join(output_path, f"P_values_{method_name.replace(' ', '_')}.csv"))
    # df_e.to_csv(os.path.join(output_path, f"Effect_sizes_{method_name.replace(' ', '_')}.csv"))
    # print(f"Effect size table saved for {method_name}.")




Normality ratio for All Windows: 15.3% → Wilcoxon Test

Normality ratio for Method 2: 18.9% → Wilcoxon Test

===== ALL WINDOWS =====
Non-significant features (p ≥ 0.05):
HRV_MeanNN: 25/50
HRV_SDNN: 6/50
HRV_LFn: 15/50
HRV_HFn: 12/50
HRV_LFHF: 11/50
HRV_VLF: 40/50
HRV_LF: 11/50
HRV_HF: 11/50
HRV_ApEn: 19/50
HRV_SampEn: 18/50
HRV_DFA_alpha1: 5/50
HRV_DFA_alpha2: 12/50
HRV_SD1: 6/50
HRV_SD2: 7/50
HRV_SD1SD2: 6/50
Effect size table saved for All Windows.

===== METHOD 2 =====
Non-significant features (p ≥ 0.05):
HRV_MeanNN: 35/50
HRV_SDNN: 4/50
HRV_LFn: 9/50
HRV_HFn: 9/50
HRV_LFHF: 9/50
HRV_VLF: 33/50
HRV_LF: 9/50
HRV_HF: 17/50
HRV_ApEn: 22/50
HRV_SampEn: 17/50
HRV_DFA_alpha1: 7/50
HRV_DFA_alpha2: 8/50
HRV_SD1: 3/50
HRV_SD2: 3/50
HRV_SD1SD2: 4/50
Effect size table saved for Method 2.
